# Tool-Calling Agent: Pengembangan, Evaluasi, dan Deployment Agen AI

Notebook ini merupakan alur lengkap (*end-to-end driver*) untuk membangun, menguji, mengevaluasi, dan mendeploy agen cerdas berbasis pemanggilan alat (**Tool-Calling Agent**) menggunakan ekosistem **Databricks Mosaic AI Agent Framework** dan **MLflow**.

### Ringkasan Proses:
1. **Instalasi Pustaka & Setup**: Menginstal `databricks-agents`, `mlflow`, `databricks-openai`, serta menginisialisasi lingkungan lab dari `Lab_Setup`.
2. **Definisi Agen (`agent.py`)**: Menulis kode agen berbasis `mlflow.pyfunc.ResponsesAgent` yang memanfaatkan model LLM dan terhubung dengan **Unity Catalog Functions** serta **Vector Search Index**.
3. **Pengujian Lokal**: Menguji respons agen baik dalam mode *batch* (`predict`) maupun *streaming* (`predict_stream`).
4. **Logging Model ke MLflow**: Mencatat agen sebagai model MLflow dengan konfigurasi dependensi resource untuk *automatic authentication passthrough*.
5. **Evaluasi Kualitas Agen**: Menguji akurasi dan kepatuhan agen menggunakan **MLflow GenAI Evaluation** (metrik Relevansi dan Keamanan).
6. **Validasi Pre-Deployment**: Menguji fungsi inferensi model dari artefak run MLflow di lingkungan terisolasi (`uv`).
7. **Pendaftaran ke Unity Catalog**: Mendaftarkan model agen ke **Unity Catalog Model Registry** dengan tata kelola terpusat.
8. **Deployment ke Model Serving**: Mempublikasikan agen ke endpoint REST API produksi menggunakan `databricks.agents.deploy`.


In [0]:
# Menginstal dependensi yang dibutuhkan: backoff, databricks-openai, uv, databricks-agents, dan mlflow
%pip install -U -qqqq backoff databricks-openai uv databricks-agents mlflow-skinny[databricks]
# Merestart kernel Python agar library yang baru diinstal dapat langsung dimuat
dbutils.library.restartPython()

In [0]:
%run ../Includes/Lab_Setup

### PENTING: Konfigurasi Namespace Unity Catalog
**TODO**: Pastikan konfigurasi pada sel kode `agent.py` di bawah menggunakan nama katalog dan skema yang sesuai dengan sesi lab Anda:

- `UC_TOOL_NAMES = ["<KATALOG>.<SKEMA>.*"]` (fungsi-fungsi Unity Catalog yang digunakan sebagai tools)
- `index_name="<KATALOG>.<SKEMA>.dphone_vs_index"` (nama index Vector Search yang dibuat pada lab 1.2)


## Langkah 1: Mendefinisikan Kode Agen (agent.py)
Kita mendefinisikan seluruh kode implementasi agen di dalam satu sel kode di bawah ini, lalu menyimpannya ke file lokal `agent.py` menggunakan magic command `%%writefile` agar dapat dicatat (*logged*) dan dideploy oleh MLflow.


In [0]:
%%writefile agent.py
import json
from typing import Any, Callable, Generator, Optional
from uuid import uuid4
import warnings

import backoff
import mlflow
import openai
from databricks.sdk import WorkspaceClient
from databricks_openai import UCFunctionToolkit, VectorSearchRetrieverTool
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
from openai import OpenAI
from pydantic import BaseModel
from unitycatalog.ai.core.base import get_uc_function_client

###############################################################################
## 1. Konfigurasi Endpoint Model LLM dan Prompt Sistem
###############################################################################
LLM_ENDPOINT_NAME = "databricks-gpt-oss-120b"

# Prompt sistem opsional untuk mengarahkan persona atau batasan perilaku agen
SYSTEM_PROMPT = """"""


###############################################################################
## 2. Definisi Tools untuk Agen (Unity Catalog Functions & Vector Search)
###############################################################################
class ToolInfo(BaseModel):
    """
    Representasi data sebuah Tool untuk agen:
    - name (str): Nama unik tool.
    - spec (dict): Deskripsi skema JSON tool (sesuai format OpenAI Tool Calling).
    - exec_fn (Callable): Fungsi Python yang mengeksekusi logika tool saat dipanggil LLM.
    """
    name: str
    spec: dict
    exec_fn: Callable


def create_tool_info(tool_spec, exec_fn_param: Optional[Callable] = None):
    """Fungsi pembantu untuk membungkus fungsi UC menjadi objek ToolInfo."""
    tool_spec["function"].pop("strict", None)
    tool_name = tool_spec["function"]["name"]
    udf_name = tool_name.replace("__", ".")

    # Wrapper yang menerima parameter dari LLM dan meneruskannya ke UC function client
    def exec_fn(**kwargs):
        function_result = uc_function_client.execute_function(udf_name, kwargs)
        if function_result.error is not None:
            return function_result.error
        else:
            return function_result.value
    return ToolInfo(name=tool_name, spec=tool_spec, exec_fn=exec_fn_param or exec_fn)


TOOL_INFOS = []

# A. Mendaftarkan fungsi-fungsi Unity Catalog (UDF) sebagai Tools
UC_TOOL_NAMES = [f"workspace.genai_course.*"]

uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
uc_function_client = get_uc_function_client()
for tool_spec in uc_toolkit.tools:
    TOOL_INFOS.append(create_tool_info(tool_spec))


# B. Mendaftarkan Databricks Vector Search Index sebagai Retriever Tool
VECTOR_SEARCH_TOOLS = []
VECTOR_SEARCH_TOOLS.append(
        VectorSearchRetrieverTool(
            index_name="workspace.genai_course.dphone_vs_index",
            # Opsional: tentukan deskripsi tool untuk membantu LLM memilih index ini
            # tool_description=""
        )
    )
for vs_tool in VECTOR_SEARCH_TOOLS:
    TOOL_INFOS.append(create_tool_info(vs_tool.tool, vs_tool.execute))


###############################################################################
## 3. Implementasi Kelas Agen Berbasis ResponsesAgent (Tool-Calling Agent)
###############################################################################
class ToolCallingAgent(ResponsesAgent):
    """
    Kelas utama agen cerdas yang mengimplementasikan protokol ResponsesAgent MLflow.
    """

    def __init__(self, llm_endpoint: str, tools: list[ToolInfo]):
        """Inisialisasi agen dengan model serving endpoint dan kumpulan tools."""
        self.llm_endpoint = llm_endpoint
        self.workspace_client = WorkspaceClient()
        self.model_serving_client: OpenAI = (
            self.workspace_client.serving_endpoints.get_open_ai_client()
        )
        self._tools_dict = {tool.name: tool for tool in tools}

    def get_tool_specs(self) -> list[dict]:
        """Mengembalikan spesifikasi tools dalam format yang diharapkan oleh OpenAI."""
        return [tool_info.spec for tool_info in self._tools_dict.values()]

    @mlflow.trace(span_type=SpanType.TOOL)
    def execute_tool(self, tool_name: str, args: dict) -> Any:
        """Mengeksekusi tool yang dipanggil oleh model dengan argumen yang diberikan (terlacak di MLflow trace)."""
        return self._tools_dict[tool_name].exec_fn(**args)

    def call_llm(self, messages: list[dict[str, Any]]) -> Generator[dict[str, Any], None, None]:
        """Memanggil model LLM endpoint secara streaming dengan daftar tools yang tersedia."""
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="PydanticSerializationUnexpectedValue")
            for chunk in self.model_serving_client.chat.completions.create(
                model=self.llm_endpoint,
                messages=to_chat_completions_input(messages),
                tools=self.get_tool_specs(),
                stream=True,
            ):
                chunk_dict = chunk.to_dict()
                if len(chunk_dict.get("choices", [])) > 0:
                    yield chunk_dict

    def handle_tool_call(
        self,
        tool_call: dict[str, Any],
        messages: list[dict[str, Any]],
    ) -> ResponsesAgentStreamEvent:
        """Mengeksekusi permintaan pemanggilan tool dan mengembalikan event output tool ke histori percakapan."""
        try:
            args = json.loads(tool_call.get("arguments"))
        except Exception as e:
            args = {}
        result = str(self.execute_tool(tool_name=tool_call["name"], args=args))

        tool_call_output = self.create_function_call_output_item(tool_call["call_id"], result)
        messages.append(tool_call_output)
        return ResponsesAgentStreamEvent(type="response.output_item.done", item=tool_call_output)

    def call_and_run_tools(
        self,
        messages: list[dict[str, Any]],
        max_iter: int = 10,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        """Loop penalaran (ReAct loop): memanggil LLM, mengeksekusi tool bila diminta, dan mengulang hingga respons final."""
        for _ in range(max_iter):
            last_msg = messages[-1]
            if last_msg.get("role", None) == "assistant":
                return
            elif last_msg.get("type", None) == "function_call":
                yield self.handle_tool_call(last_msg, messages)
            else:
                yield from output_to_responses_items_stream(
                    chunks=self.call_llm(messages), aggregator=messages
                )

        yield ResponsesAgentStreamEvent(
            type="response.output_item.done",
            item=self.create_text_output_item("Max iterations reached. Stopping.", str(uuid4())),
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        """Metode inferensi non-streaming (batch). Mengembalikan seluruh respons setelah komputasi selesai."""
        session_id = None
        if request.custom_inputs and "session_id" in request.custom_inputs:
            session_id = request.custom_inputs.get("session_id")
        elif request.context and request.context.conversation_id:
            session_id = request.context.conversation_id

        if session_id:
            mlflow.update_current_trace(
                metadata={
                    "mlflow.trace.session": session_id,
                }
            )

        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(self, request: ResponsesAgentRequest) -> Generator[ResponsesAgentStreamEvent, None, None]:
        """Metode inferensi streaming. Mengirimkan token dan event output secara real-time saat diproses."""
        session_id = None
        if request.custom_inputs and "session_id" in request.custom_inputs:
            session_id = request.custom_inputs.get("session_id")
        elif request.context and request.context.conversation_id:
            session_id = request.context.conversation_id

        if session_id:
            mlflow.update_current_trace(
                metadata={
                    "mlflow.trace.session": session_id,
                }
            )

        messages = to_chat_completions_input([i.model_dump() for i in request.input])
        if SYSTEM_PROMPT:
            messages.insert(0, {"role": "system", "content": SYSTEM_PROMPT})
        yield from self.call_and_run_tools(messages=messages)


###############################################################################
## 4. Inisialisasi Model dan Registrasi ke MLflow
###############################################################################
# Mengaktifkan autologging untuk tracing otomatis setiap pemanggilan OpenAI SDK
mlflow.openai.autolog()
AGENT = ToolCallingAgent(llm_endpoint=LLM_ENDPOINT_NAME, tools=TOOL_INFOS)
mlflow.models.set_model(AGENT)


## Langkah 2: Menguji Agen secara Lokal
Melakukan interaksi uji coba dengan agen untuk memvalidasi output dan alur pemanggilan alat.
Karena metode di dalam `ResponsesAgent` telah dipasangi tracing, setiap langkah yang diambil agen dan setiap pemanggilan LLM via OpenAI SDK akan terlacak otomatis (*autologged*) di antarmuka MLflow Tracing.


In [0]:
# Mengimpor instance agen yang baru didefinisikan dari file agent.py
from agent import AGENT

# Menguji inferensi agen secara langsung (batch) dengan pertanyaan seputar spesifikasi produk
AGENT.predict(
    {"input": [{"role": "user", "content": "What is the screen size of dPhone D1 Fold?"}], "custom_inputs": {"session_id": "test-session-123"}},
)

In [0]:
# Menguji inferensi agen dalam mode streaming (token demi token) dan menampilkan setiap chunk output
for chunk in AGENT.predict_stream(
    {"input": [{"role": "user", "content": "What is the screen size of dPhone D1 Fold?"}], "custom_inputs": {"session_id": "test-session-123"}}
):
    print(chunk.model_dump(exclude_none=True))

## Langkah 3: Mencatat (Log) Agen sebagai Model MLflow
Menentukan dependensi sumber daya (*Databricks resources*) yang diperlukan untuk otorisasi otomatis (*automatic authentication passthrough*) saat agen dideploy ke Model Serving:
- Endpoint model serving LLM (`LLM_ENDPOINT_NAME`)
- Objek index Vector Search (`VECTOR_SEARCH_TOOLS`)
- Fungsi-fungsi Unity Catalog (`DatabricksFunction`)

Model dicatat menggunakan pendekatan *Models from Code* dari file `agent.py`.


In [0]:
# Menentukan dependensi resource Databricks untuk automatic auth passthrough saat deployment
import mlflow
from agent import LLM_ENDPOINT_NAME, VECTOR_SEARCH_TOOLS, uc_toolkit
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint
from pkg_resources import get_distribution

# Mendaftarkan Serving Endpoint model LLM sebagai resource
resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME)]

# Mendaftarkan resource Vector Search Index
for tool in VECTOR_SEARCH_TOOLS:
    resources.extend(tool.resources)

# Mendaftarkan fungsi-fungsi Unity Catalog yang digunakan agen
for tool in uc_toolkit.tools:
    udf_name = tool.get("function", {}).get("name", "").replace("__", ".")
    resources.append(DatabricksFunction(function_name=udf_name))

# Contoh input skema untuk dokumentasi model di MLflow
input_example = {
    "input": [
        {
            "role": "user",
            "content": "What is dPhone D1 Fold?"
        }
    ],
    "custom_inputs": {
        "session_id": "test-session"
    }
}

# Mencatat model agen ke dalam run MLflow Tracking
with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        input_example=input_example,
        pip_requirements=[
            "databricks-openai",
            "backoff",
            f"databricks-connect=={get_distribution('databricks-connect').version}",
        ],
        resources=resources,
    )

## Langkah 4: Evaluasi Agen Menggunakan Agent Evaluation
Menjalankan pengujian kualitas agen menggunakan framework **Mosaic AI Agent Evaluation** dan metrik penilaian otomatis (*predefined LLM judge scorers*):
- `RelevanceToQuery`: Menilai apakah respons menjawab pertanyaan pengguna dengan tepat.
- `Safety`: Memastikan respons aman dan bebas dari konten berbahaya/kebocoran data.

Hasil evaluasi dapat ditinjau langsung di MLflow UI.


In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, Safety, RetrievalRelevance, RetrievalGroundedness

# Dataset pengujian kualitas respons agen
eval_dataset = [
    {
        "inputs": {
            "input": [
                {
                    "role": "user",
                    "content": "What is dPhone D1 Fold?"
                }
            ]
        },
        "expected_response": None
    }
]

# Menjalankan evaluasi GenAI menggunakan fungsi penilaian MLflow
eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: AGENT.predict({"input": input, "custom_inputs": {"session_id": "evaluation-session"}}),
    scorers=[RelevanceToQuery(), Safety()], # Tambahkan metrik penilaian lain bila diperlukan
)

# Tinjau hasil evaluasi pada tampilan MLflow UI

## Langkah 5: Melakukan Validasi Pre-Deployment Agen
Sebelum mendaftarkan dan mendeploy agen, kita memvalidasi fungsionalitas inferensi model dari artefak run MLflow di lingkungan paket terisolasi menggunakan API `mlflow.models.predict()`.


In [0]:
# Menjalankan uji prediksi pre-deployment menggunakan environment manager "uv"
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"input": [{"role": "user", "content": "Hello!"}], "custom_inputs": {"session_id": "validation-session"}},
    env_manager="uv",
)

## Langkah 6: Mendaftarkan Model ke Unity Catalog
Mendaftarkan model agen yang telah tervalidasi ke **Unity Catalog Model Registry** dengan tata kelola versi, kontrol akses (*ACL*), dan *lineage* terpusat.


In [0]:
# Menyetel target registry URI ke Unity Catalog Databricks
mlflow.set_registry_uri("databricks-uc")

# Menentukan namespace katalog, skema, dan nama model UC
catalog = course.catalog
schema = course.schema
model_name = "dphone_agent"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# Mendaftarkan model agen ke Unity Catalog
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME
)

## Langkah 7: Mendeploy Agen ke Model Serving
Mempublikasikan model terdaftar di Unity Catalog menjadi endpoint REST API terkelola menggunakan SDK `databricks.agents`.
Endpoint ini dapat diintegrasikan ke aplikasi web, chat frontend, atau sistem backend lainnya.


In [0]:
from databricks import agents

# Mendeploy model agen ke Databricks Model Serving
# Catatan: parameter scale_to_zero=True dapat ditambahkan untuk menghemat biaya komputasi (tidak disarankan untuk produksi)
agents.deploy(UC_MODEL_NAME, uc_registered_model_info.version, tags = {"endpointSource": "playground"})

## Langkah Selanjutnya
Setelah agen berhasil dideploy:
1. Anda dapat berinteraksi langsung dengan agen melalui **AI Playground** di Databricks Workspace untuk pengujian antarmuka.
2. Membagikan endpoint kepada para pemangku kepentingan (*SMEs*) untuk mendapatkan umpan balik langsung.
3. Mengintegrasikan endpoint REST API agen ke dalam aplikasi produksi atau pipeline analitik Anda.
